In [ ]:
# Install required libraries (run once in a notebook)
!pip install transformers datasets requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import requests
from transformers import GPT2TokenizerFast

# ----------------------------------------------------------------------
# Configuration - to hold all hyperparameters
#    Used by: SelfAttention, FFN, TransformerBlock, GPT2
#    Why: centralises settings, easy to change model size

# ----------------------------------------------------------------------
class Config:
    def __init__(self, vocab_size=50257, max_seq_length=128, embed_size=768,
                 num_layers=12, num_heads=12, dropout=0.1):
        # vocab_size: size of token vocabulary (GPT-2 uses 50257)
        self.vocab_size = vocab_size
        # max_seq_length: maximum number of tokens the model can process at once
        self.max_seq_length = max_seq_length
        # embed_size: dimension of token embeddings (also called d_model)
        self.embed_size = embed_size
        # num_layers: number of transformer blocks (decoder layers)
        self.num_layers = num_layers
        # num_heads: number of attention heads in multi-head attention
        self.num_heads = num_heads
        # dropout: dropout probability used throughout the model
        self.dropout = dropout

# ----------------------------------------------------------------------
# Multi-Head Causal Self-Attention (fixed scaling)
#    Used by: TransformerBlock (as the attention component)
#    Why: allows the model to weigh different positions in the input sequence,
#         while the causal mask prevents looking at future tokens (autoregressive).

# ----------------------------------------------------------------------
class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Ensure embed_size is divisible by num_heads (so each head gets equal dimension)
        assert config.embed_size % config.num_heads == 0
        self.num_heads = config.num_heads
        # head_dim = d_k = embed_size / num_heads
        self.head_dim = config.embed_size // config.num_heads

        # Linear layers to project input to queries, keys, values
        # Input: (batch, seq_len, embed_size) -> Output same shape
        self.W_q = nn.Linear(config.embed_size, config.embed_size)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)

        # Output projection (combines heads back to embed_size)
        self.out_proj = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

        # Create causal (lower triangular) mask to prevent looking at future tokens
        # Shape: (max_seq_length, max_seq_length) with 1s on and below diagonal
        mask = torch.tril(torch.ones(config.max_seq_length, config.max_seq_length))
        # Reshape to (1, 1, max_seq_len, max_seq_len) for broadcasting over batch and heads
        self.register_buffer('mask', mask.view(1, 1, config.max_seq_length, config.max_seq_length))

    def forward(self, x):
        # x comes from the previous layer (either token embeddings or previous TransformerBlock
        # x shape: (batch_size, seq_len, embed_size)
        batch, seq_len, embed_dim = x.shape

        # ---- Step 1: Compute Q, K, V ----
        # Linear projection + reshape to separate heads
        # W_q(x) -> (batch, seq_len, embed_size)
        # view(...) -> (batch, seq_len, num_heads, head_dim)
        # transpose(1,2) -> (batch, num_heads, seq_len, head_dim)
                # Each becomes (batch, num_heads, seq_len, head_dim)
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)


        # ---- Step 2: Scaled dot-product attention ----
        # Compute attention scores: Q · K^T / sqrt(head_dim)
        # Q shape: (batch, num_heads, seq_len, head_dim)
        # K.transpose(-2,-1) shape: (batch, num_heads, head_dim, seq_len)
        # Result (attn_scores) shape: (batch, num_heads, seq_len, seq_len)
        # Example: if seq_len=3, then cell (i,j) = dot product of Q_i and K_j
        attn_scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # ---- Step 3: Apply causal mask ----
        # mask[:, :, :seq_len, :seq_len] shape: (1, 1, seq_len, seq_len)
        # Where mask is 0 (upper triangular), set attn_scores to -inf so softmax ignores them
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len] == 0, float('-inf')
        )

        # ---- Step 4: Softmax and dropout ----
        # Softmax along last dimension (over keys) -> attention probabilities
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        # ---- Step 5: Weighted sum of values ---
        # attn_probs @ V: (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, head_dim)
        # -> (batch, num_heads, seq_len, head_dim)
        context = attn_probs @ V

        # ---- Step 6: Combine heads --
        # transpose(1,2) -> (batch, seq_len, num_heads, head_dim)
        # contiguous().view(...) -> (batch, seq_len, embed_size)
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, embed_dim)

        # Final linear projection
        output = self.out_proj(context)

        # Apply dropout again (supposely standard in GPT-2)
        return self.dropout(output)

# ----------------------------------------------------------------------
# Feed-Forward Network --> expands to 4*embed_size and bach
#    Used by: TransformerBlock (after attention)
#    Why: adds non‑linearity and capacity; each position processed independently.
#         Expands to 4*embed_size then back – standard in transformers.
# ----------------------------------------------------------------------
class FFN(nn.Module):
    def __init__(self, config):

        super().__init__()
        # First linear layer: embed_size -> 4*embed_size
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        # GELU activation (smoother than ReLU, used in GPT-2)
        self.gelu = nn.GELU()
        # Second linear layer: 4*embed_size -> embed_size
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # x comes from the layer normalisation after attention
        # x shape: (batch, seq_len, embed_size)
        x = self.fc2(self.gelu(self.fc1(x))) # (batch, seq_len, embed_size)
        return self.dropout(x)

# ----------------------------------------------------------------------
# Transformer Block (Pre-LN + residual)
#    Used by: GPT2 model (stacked num_layers times)
#    Why: pre‑LN improves training stability; residuals help gradients flow.
# ----------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        # LayerNorm before attention (Pre-LN, improves training stability)
        self.ln1 = nn.LayerNorm(config.embed_size) # norm before attention
        self.attn = SelfAttention(config)  # multi‑head causal attention
        # LayerNorm before MLP
        self.ln2 = nn.LayerNorm(config.embed_size) # norm before FFN
        self.mlp = FFN(config) # feed‑forward network

    def forward(self, x):
        # Residual connection around attention - add original x to the output of attention
        x = x + self.attn(self.ln1(x))
        # Residual connection around MLP - add to the output of the FFN
        x = x + self.mlp(self.ln2(x))
        return x

# ---------------------------------------------------------------
# GPT-2 Model -- token + position embeddings, stack of blocks, final LN & weight tying
#    Used by: training loop and generation function
#    Why: combines token + position embeddings, runs through transformer blocks,
#         then projects to vocabulary size (weight tying).
# ---------------------------------------------------------
class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # Token embedding: maps token IDs (0..vocab_size-1) to embed_size vectors
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size)
        # Position embedding: learns a vector for each position (0..max_seq_len-1)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        # Stack of transformer blocks
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.num_layers)])
        # Final LayerNorm (after all blocks, before output projection)
        self.ln_f = nn.LayerNorm(config.embed_size)

    def forward(self, input_ids):
        # input_ids shape: (batch, seq_len)
        batch, seq_len = input_ids.shape
        # Create position indices: 0,1,2,...,seq_len-1
        #  unsqueeze(0) adds batch dimension -> (1, seq_len)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        # Add token embeddings and position embeddings
        # token_embed(input_ids): (batch, seq_len, embed_size)
        # pos_embed(positions):   (1, seq_len, embed_size) -> broadcasted to batch
        x = self.token_embed(input_ids) + self.pos_embed(positions)
        x = self.dropout(x)
        # Pass through all transformer blocks
        x = self.blocks(x) # shape unchanged: (batch, seq_len, embed_size
        x = self.ln_f(x)

        # Weight tying: output logits = x @ (token_embedding weights)^T
        # token_embed.weight shape: (vocab_size, embed_size)
        # x shape: (batch, seq_len, embed_size)
        # result logits shape: (batch, seq_len, vocab_size)
        logits = x @ self.token_embed.weight.T
        return logits


In [ ]:

# ----------------------------------------------------------------------
# Download and tokenize Don Quijote
# Why: we need training data. We use Project Gutenberg for a free book
# ----------------------------------------------------------------------
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
response = requests.get(url)
text = response.text

# Load GPT-2 tokenizer (subword tokenizer)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
# GPT-2 doesn't have a padding token by default; set it to EOS token
tokenizer.pad_token = tokenizer.eos_token
# Encode whole text to a list of token IDs
tokens = tokenizer.encode(text)
# Convert to PyTorch tensor of type long (int64)
data = torch.tensor(tokens, dtype=torch.long)

# ----------------------------------------------------------------------
# Dataset and DataLoader – create sliding window pairs (x, y)
#    Used by: training loop
#    Why: we need to feed the model (x, y) where y = x shifted by one token.
#         This is the standard language modelling task (next token prediction).
# ----------------------------------------------------------------------
SEQ_LENGTH = 128   # corrected spelling

class QuijoteDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data  # full 1D tensor of token IDs
        self.seq_length = seq_length

    def __len__(self):
        # Number of possible starting indices for a sequence of length seq_length
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        # Input tokens: positions idx .. idx+seq_length-1
        x = self.data[idx: idx + self.seq_length]
        # Target tokens: shift by 1 (predict next token)
        y = self.data[idx + 1: idx + self.seq_length + 1]
        return x, y

dataset = QuijoteDataset(data, SEQ_LENGTH)
# DataLoader: shuffles, batches (batch_size=16), parallel workers=0
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

# ----------------------------------------------------------------------
# Instantiate model, optimizer, device
#    Why: we choose a smaller config for quick training on CPU.
#         Adam is a standard optimiser for transformers.
# ----------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create a smaller configuration for fast training on CPU
config = Config(
    vocab_size=tokenizer.vocab_size,    # 50257
    max_seq_length=SEQ_LENGTH,          # 128
    embed_size=128,                     # reduced from 768
    num_layers=4,                       # reduced from 12
    num_heads=4,                        # reduced from 12
    dropout=0.1
)

model = GPT2(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4) # learning rate 0.0003

# ----------------------------------------------------------------------
# Training function
#    Why: loops over the data, does forward pass, computes cross‑entropy loss,
#         backpropagates, updates weights.
#         The loss is computed on flattened predictions vs flattened targets.
# ----------------------------------------------------------------------
def train(model, loader, optimizer, epochs=10):
    model.train() # set model to training mode (enables dropout)
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)  # move batch to GPU/CPU
            optimizer.zero_grad()      # reset gradients

            logits = model(x)   # forward pass: (batch, seq_len, vocab_size)
            # Compute cross-entropy loss
            # logits.view(-1, logits.size(-1)) -> (batch*seq_len, vocab_size)
            # y.view(-1) -> (batch*seq_len)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()      # backpropagate
            optimizer.step()        # update weights
            total_loss += loss.item()

            if i % 500 == 0:
                print(f"Epoch {epoch+1}, step {i}, loss: {loss.item():.4f}")

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

# Train for 2 epochs
train(model, dataloader, optimizer, epochs=2)

# ----------------------------------------------------------------------
# Text generation
# ----------------------------------------------------------------------
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    model.eval()  # switch to evaluation mode (no dropout --> no randomness from dropout during inference)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Encode prompt to token IDs, shape (1, len(prompt))
    for _ in range(max_new_tokens):
        # Crop to last SEQ_LENGTH tokens (so model never sees more than it was trained on)
        # Only keep the last SEQ_LENGTH tokens (otherwise we'd exceed the model's max length)
        input_ids_crop = input_ids[:, -SEQ_LENGTH:]
        with torch.no_grad():
            logits = model(input_ids_crop)   # (1, seq_len, vocab_size)
        # Get logits for the last token only: shape (vocab_size,)
        next_logits = logits[0, -1, :] / temperature   # divide by temperature (higher = more random)
        probs = F.softmax(next_logits, dim=-1)      # convert to probabilities

        next_id = torch.multinomial(probs, num_samples=1).unsqueeze(0)  # sample one token, shape (1,1)
        input_ids = torch.cat([input_ids, next_id], dim=1) # append to the sequence

        # Decode the full sequence back to text
    return tokenizer.decode(input_ids[0])



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (860015 > 1024). Running this sequence through the model will result in indexing errors


Using device: cuda
Epoch 1, step 0, loss: 78.7495
Epoch 1, step 500, loss: 9.0952
Epoch 1, step 1000, loss: 6.4651
Epoch 1, step 1500, loss: 5.9960
Epoch 1, step 2000, loss: 5.4057
Epoch 1, step 2500, loss: 5.3085
Epoch 1, step 3000, loss: 5.0404
Epoch 1, step 3500, loss: 4.9480
Epoch 1, step 4000, loss: 4.6636
Epoch 1, step 4500, loss: 4.3700
Epoch 1, step 5000, loss: 4.4398
Epoch 1, step 5500, loss: 4.3135
Epoch 1, step 6000, loss: 4.2003
Epoch 1, step 6500, loss: 4.4173
Epoch 1, step 7000, loss: 4.1058
Epoch 1, step 7500, loss: 4.1907
Epoch 1, step 8000, loss: 3.9650
Epoch 1, step 8500, loss: 4.0121
Epoch 1, step 9000, loss: 3.8927
Epoch 1, step 9500, loss: 3.7231
Epoch 1, step 10000, loss: 3.6942
Epoch 1, step 10500, loss: 3.8194
Epoch 1, step 11000, loss: 3.6828
Epoch 1, step 11500, loss: 3.6140
Epoch 1, step 12000, loss: 3.6139
Epoch 1, step 12500, loss: 3.7414
Epoch 1, step 13000, loss: 3.7628
Epoch 1, step 13500, loss: 3.5154
Epoch 1, step 14000, loss: 3.5867
Epoch 1, step 1450

In [ ]:
###====================
## USAGE EXAMPLE 1
#==============

prompt = "Un estudiante de doctorado"
output = generate(model, tokenizer, prompt, max_new_tokens=50)
print("\nGenerated text:\n", output)


Generated text:
 Un estudiante de doctorado, donde podría ser gobernador; y entre
aquí llaman de diez o menesteros y descuidos; no se me paseó en ello, que ya por
discreto


In [ ]:

##====================
## USAGE EXAMPLE 2
#==============
prompt = "Don Quijote dijo:"
generated = generate(model, tokenizer, prompt, max_new_tokens=100)
print(generated)


Don Quijote dijo:

— Estas amolas son dos leguas.

Apartóse don Antonio así en los que estaban, bajó, y, de modo queerse la llevaban
andaba la mejor fortuna, cuando los demás nos suspiros de nuevo
que salía del bachiller—, conforme aquel trance adelantado, quién la mejor


In [ ]:
##====================
## USAGE EXAMPLE 3
#==============
prompt = "Don Quijote said to Ophelia:"
generated = generate(model, tokenizer, prompt, max_new_tokens=100)
print("example 3 3:\n", generated)

example 3 3:
 Don Quijote said to Ophelia:
 Tú has de sí; si nuestro amo le pusiere a los labios de la flojares
azmenoscabo, de poner contra unos libras, arteseto a la cama, es engañamiento y
del aldea y aun las pajes de tus cosas es en que dice, creyendo del
moro de su presencia, de quien vedu


#SAVE & LOAD MODEL

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
##====================
## SAVE TRAINNED MODEL
##=====================

save_folder = "/content/drive/MyDrive/Colab_notebooks/Tec_hws/assg_p2_gpt_6jun"
os.makedirs(save_folder, exist_ok=True)


torch.save(model.state_dict(), os.path.join(save_folder, "model_weights.pth"))
torch.save(config, os.path.join(save_folder, "config.pth"))




# Or save the entire model (including architecture) – but state_dict is recommended
# torch.save(model, "full_model.pth")

print("Model saved successfully in the current directory.")

In [ ]:
##================
## LOAD SAVED MODEL
##=================
# Recreate the model with the same config
loaded_config = torch.load("config.pth")
model = GPT2(loaded_config).to(device)

# Load the saved weights
model.load_state_dict(torch.load("model_weights.pth", map_location=device))
model.eval()   # or model.train() if continuing training

#LOADING ANOTHER CORPUS OF TEXT


In [ ]:
# ----------------------------------------------------------------------
# Download and tokenize two texts
# ----------------------------------------------------------------------
# 1. Don Quijote (Spanish)
url_quijote = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
response_quijote = requests.get(url_quijote)
text_quijote = response_quijote.text

# 2. Another text – for example, Pride and Prejudice (English)
url_pride = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
response_pride = requests.get(url_pride)
text_pride = response_pride.text

# Optional: clean both texts (remove Project Gutenberg headers/footers)
# But for simplicity, we'll use them as is.

# Load tokenizer (still English‑only!)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenise both texts separately
tokens_quijote = tokenizer.encode(text_quijote)
tokens_pride = tokenizer.encode(text_pride)

# Combine them: you can concatenate, interleave, or balance the dataset
# Option A: simple concatenation (all Quijote first, then Pride)
combined_tokens = tokens_quijote + tokens_pride

# Option B: shuffle after concatenation (random mix)
import random
combined_tokens = tokens_quijote + tokens_pride
random.shuffle(combined_tokens)   # random order of sentences/tokens – not perfect, but works

# Option C: take a balanced subset (e.g., 1 million tokens from each)
# combined_tokens = tokens_quijote[:1_000_000] + tokens_pride[:1_000_000]

# Convert to tensor
data = torch.tensor(combined_tokens, dtype=torch.long)

# The rest of the code (Dataset, DataLoader, training) remains exactly the same.

# ----------------------------------------------------------------------
# Dataset and DataLoader – create sliding window pairs (x, y)
#    Used by: training loop
#    Why: we need to feed the model (x, y) where y = x shifted by one token.
#         This is the standard language modelling task (next token prediction).
# ----------------------------------------------------------------------
SEQ_LENGTH = 128   # corrected spelling

class MixedDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data  # full 1D tensor of token IDs
        self.seq_length = seq_length

    def __len__(self):
        # Number of possible starting indices for a sequence of length seq_length
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        # Input tokens: positions idx .. idx+seq_length-1
        x = self.data[idx: idx + self.seq_length]
        # Target tokens: shift by 1 (predict next token)
        y = self.data[idx + 1: idx + self.seq_length + 1]
        return x, y

dataset = MixedDataset(data, SEQ_LENGTH)
# DataLoader: shuffles, batches (batch_size=16), parallel workers=0
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

# ----------------------------------------------------------------------
# Instantiate model, optimizer, device
#    Why: we choose a smaller config for quick training on CPU.
#         Adam is a standard optimiser for transformers.
# ----------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create a smaller configuration for fast training on CPU
config = Config(
    vocab_size=tokenizer.vocab_size,    # 50257
    max_seq_length=SEQ_LENGTH,          # 128
    embed_size=128,                     # reduced from 768
    num_layers=4,                       # reduced from 12
    num_heads=4,                        # reduced from 12
    dropout=0.1
)

model = GPT2(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4) # learning rate 0.0003

# ----------------------------------------------------------------------
# Training function
#    Why: loops over the data, does forward pass, computes cross‑entropy loss,
#         backpropagates, updates weights.
#         The loss is computed on flattened predictions vs flattened targets.
# ----------------------------------------------------------------------
def train(model, loader, optimizer, epochs=10):
    model.train() # set model to training mode (enables dropout)
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)  # move batch to GPU/CPU
            optimizer.zero_grad()      # reset gradients

            logits = model(x)   # forward pass: (batch, seq_len, vocab_size)
            # Compute cross-entropy loss
            # logits.view(-1, logits.size(-1)) -> (batch*seq_len, vocab_size)
            # y.view(-1) -> (batch*seq_len)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()      # backpropagate
            optimizer.step()        # update weights
            total_loss += loss.item()

            if i % 500 == 0:
                print(f"Epoch {epoch+1}, step {i}, loss: {loss.item():.4f}")

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

# Train for 1 epochs
train(model, dataloader, optimizer, epochs=1)

# ----------------------------------------------------------------------
# Text generation
# ----------------------------------------------------------------------
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    model.eval()  # switch to evaluation mode (no dropout --> no randomness from dropout during inference)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Encode prompt to token IDs, shape (1, len(prompt))
    for _ in range(max_new_tokens):
        # Crop to last SEQ_LENGTH tokens (so model never sees more than it was trained on)
        # Only keep the last SEQ_LENGTH tokens (otherwise we'd exceed the model's max length)
        input_ids_crop = input_ids[:, -SEQ_LENGTH:]
        with torch.no_grad():
            logits = model(input_ids_crop)   # (1, seq_len, vocab_size)
        # Get logits for the last token only: shape (vocab_size,)
        next_logits = logits[0, -1, :] / temperature   # divide by temperature (higher = more random)
        probs = F.softmax(next_logits, dim=-1)      # convert to probabilities

        next_id = torch.multinomial(probs, num_samples=1).unsqueeze(0)  # sample one token, shape (1,1)
        input_ids = torch.cat([input_ids, next_id], dim=1) # append to the sequence

        # Decode the full sequence back to text
    return tokenizer.decode(input_ids[0])


Token indices sequence length is longer than the specified maximum sequence length for this model (860015 > 1024). Running this sequence through the model will result in indexing errors


Using device: cuda
Epoch 1, step 0, loss: 80.8917
Epoch 1, step 500, loss: 9.7402
Epoch 1, step 1000, loss: 7.4311
Epoch 1, step 1500, loss: 7.0972
Epoch 1, step 2000, loss: 6.7880
Epoch 1, step 2500, loss: 6.7271
Epoch 1, step 3000, loss: 6.5630
Epoch 1, step 3500, loss: 6.5064
Epoch 1, step 4000, loss: 6.5341
Epoch 1, step 4500, loss: 6.3164
Epoch 1, step 5000, loss: 6.3746
Epoch 1, step 5500, loss: 6.3498
Epoch 1, step 6000, loss: 6.4086
Epoch 1, step 6500, loss: 6.4422
Epoch 1, step 7000, loss: 6.4353
Epoch 1, step 7500, loss: 6.4087
Epoch 1, step 8000, loss: 6.3878
Epoch 1, step 8500, loss: 6.2783
Epoch 1, step 9000, loss: 6.3328
Epoch 1, step 9500, loss: 6.3865
Epoch 1, step 10000, loss: 6.3455
Epoch 1, step 10500, loss: 6.2891
Epoch 1, step 11000, loss: 6.3038
Epoch 1, step 11500, loss: 6.2640
Epoch 1, step 12000, loss: 6.3423
Epoch 1, step 12500, loss: 6.2908
Epoch 1, step 13000, loss: 6.3788
Epoch 1, step 13500, loss: 6.2748
Epoch 1, step 14000, loss: 6.3229
Epoch 1, step 1450

In [ ]:
###====================
## USAGE EXAMPLE 1
#==============

prompt = "Un estudiante de doctorado"
output = generate(model, tokenizer, prompt, max_new_tokens=50)
print("\nNEW 1 text:\n", output)

In [ ]:

##====================
## USAGE EXAMPLE 2
#==============
prompt = "Don Quijote dijo:"
generated = generate(model, tokenizer, prompt, max_new_tokens=100)
print("NEW 2:\n", generated)

In [ ]:
##====================
## USAGE EXAMPLE 2
#==============
prompt = "Don Quijote said to Ophelia:"
generated = generate(model, tokenizer, prompt, max_new_tokens=100)
print("NEW 3:\n", generated)

# MULTILINGUAL TOKENIZER

> Agregar bloque entrecomillado



In [ ]:
# 1. Use multilingual tokenizer
from transformers import XLMRobertaTokenizerFast

tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")
tokenizer.pad_token = tokenizer.eos_token   # if needed

# 2. Load and tokenise both Spanish and English texts (same as before)
url_quijote = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
url_pride = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
text_quijote = requests.get(url_quijote).text
text_pride = requests.get(url_pride).text

tokens_quijote = tokenizer.encode(text_quijote)
tokens_pride = tokenizer.encode(text_pride)

combined_tokens = tokens_quijote + tokens_pride
import random
random.shuffle(combined_tokens)

data = torch.tensor(combined_tokens, dtype=torch.long)

# 3. Dataset, DataLoader (unchanged)
SEQ_LENGTH = 128
dataset = QuijoteDataset(data, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 4. Config with new vocab_size
config = Config(
    vocab_size=tokenizer.vocab_size,   # now 250002
    max_seq_length=SEQ_LENGTH,
    embed_size=128,
    num_layers=4,
    num_heads=4,
    dropout=0.1
)

# 5. Model, optimizer (unchanged)
model = GPT2(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

# 6. Train (unchanged)
train(model, dataloader, optimizer, epochs=3)

# 7. Generate with mixed-language prompt (unchanged logic)
prompt = "Don Quijote said to Ophelia but in spanish and then speaking in english:"   # Spanish + English
generated = generate(model, tokenizer, prompt, max_new_tokens=100)
print(generated)